# Library imports and setting file path

In [ ]:
import os
import sys
import warnings
import pandas as pd
import geopandas as gpd

ROOT = os.path.abspath(".")
PY_SCRIPTS = os.path.join(ROOT, "py_scripts")
sys.path.insert(0, PY_SCRIPTS)

from parse_adjacency import parse_adjacency, load_adjacency
from parse_county_data import parse_county_data, load_county_data
from min_coverage import solve_min_coverage
from equitable_placement import solve_equitable
from helper_functions import STATE_FIPS_TO_NAME, plot_centers


SHP_PATH = os.path.join(ROOT, "county_data", "cb_2023_us_county_500k", "cb_2023_us_county_500k.shp")
DEMO_PATH = os.path.join(ROOT, "county_data", "county_demographics.csv")

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
warnings.filterwarnings('ignore')

# User defines desired state of interest and k value for equitable placement model

In [ ]:
STATE_NAME = "Maine"
STATE_FIPS = {name: fips for fips, name in STATE_FIPS_TO_NAME.items()}[STATE_NAME]

k = 10

# Parse county adjacency, county data, and shapefile data

In [ ]:
parse_adjacency(STATE_FIPS)
parse_county_data(DEMO_PATH, STATE_FIPS)
adj = load_adjacency(STATE_FIPS)
cdata = load_county_data(STATE_FIPS)

# Load shapefile and filter to desired state
shp = gpd.read_file(SHP_PATH)
state_shp = shp[shp["STATEFP"] == STATE_FIPS].copy()
state_shp = state_shp.to_crs(state_shp.estimate_utm_crs())

# Solve Minimum Coverage Problem

In [ ]:
centers_a, obj_a, time_a = solve_min_coverage(adj, cdata)
print(f"Min coverage: {len(centers_a)} centers, objective {obj_a}, {time_a:.4f}s")
plot_centers(state_shp, cdata, centers_a, "min coverage")

In [ ]:
df_a = pd.DataFrame(
    [
        {
            "FIPS": fips,
            "County": data["name"],
            "Population": data["population"],
            "Is_Center": fips in centers_a,
        }
        for fips, data in cdata.items()
    ]
)
display(df_a)

# Solve Equitable Placement Problem

In [ ]:
centers_b, obj_b, time_b, assigned = solve_equitable(cdata, k)
print(f"Equitable (k={k}): {len(centers_b)} centers, objective {obj_b}, {time_b:.4f}s")
plot_centers(state_shp, cdata, centers_b, f"k={k} equitable")

In [ ]:
df_b = pd.DataFrame(
    [
        {
            "FIPS": fips,
            "County": data["name"],
            "Population": data["population"],
            "Is_Center": fips in centers_b,
            "Assigned_To": assigned.get(fips),
        }
        for fips, data in cdata.items()
    ]
)
display(df_b)